# Dataset generation

Generates spiking datasets from a NEURON network you configure here. One fixed
network; states differ by a single parameter (`sahp_ainc_slow`).

Recordings warm-start from a saved network state rather than
`h.finitialize(-65)`, which would put every cell in the same artificial
zero-adaptation condition and make the whole population fire together early in
every recording.

Generation runs **in parallel** — `n_workers` NEURON processes at a time, shared
across states.

**New dataset vs continuing one.** The `session` name decides. A new name starts
fresh; an existing name resumes — raise `n_recordings` and rerun, and only the
missing indices are generated, reusing the same network and warm-start libraries.
Everything is resumable, so an interrupted run is continued the same way.

Edit **CONFIG**, then run the cells in order.


In [ ]:
%matplotlib inline
import os, sys
REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'analysis'), os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)

import dataset_nb as nb          # all the plumbing lives here
from neuron_simulation.topology import NeuronWeightParameters


In [ ]:
# ==========================================================================
# CONFIG - the only cell you need to edit. It is the single source of truth:
# the graph, the warm-start libraries and every recording are built from it.
# ==========================================================================

# ---- synaptic weight ranges (used when the graph is built) ------------------
wp = NeuronWeightParameters()
wp.within_exc_range  = (0.0010, 0.0022)   # exc weight, same cluster
wp.between_exc_range = (0.0008, 0.0016)   # exc weight, across clusters
wp.within_inh_range  = (0.0025, 0.0055)   # inh weight, same cluster
wp.between_inh_range = (0.0020, 0.0040)   # inh weight, across clusters
wp.use_lognormal     = True
wp.lognormal_sigma   = 0.5

CONFIG = {

    # Output goes to notebooks/NEURON data parallel/<session>/<state>/.
    # Change this for a new dataset; an existing session is resumed, not redone.
    'session': 'dataset_v1',

    # ---- states: the ONE parameter that differs between them ---------------
    # sahp_ainc_slow is the slow-AHP / M-current strength. Lower = weaker
    # adaptation = more excitable. Add or rename entries freely.
    'states': {'normal': 0.01, 'seizure': 0.004},
    'states_to_run': ['normal', 'seizure'],

    # ---- the network graph --------------------------------------------------
    'topology_kind': 'lognormal',            # 'lognormal' | 'discrete_hub'
    'topology': dict(
        num_clusters=50, neurons_per_cluster_range=(4, 40),
        inhibitory_probability=0.2, cluster_radius=1.0, space_size=15.0, seed=1,
        decay_sigma=3.0, max_connection_distance=6.0,
        cell_type_specific=True,
        p_ee_within=0.2, p_ee_between=0.1, p_ei_within=0.20, p_ei_between=0.08,
        p_ie_within=0.40, p_ii_within=0.50,
        within_cluster_prob=0.25, between_cluster_prob=0.06,
        ln_sigma=0.5, target_density=None, weight_params=wp,
    ),

    # ---- cell and synapse parameters ---------------------------------------
    # sahp_ainc_slow here is a placeholder; the per-state value from 'states'
    # replaces it. Everything else applies to every state.
    'build': dict(
        synapse_model='ampa_nmda', exc_tau=5.0, tau_nmda=350.0, nmda_ratio=3.0,
        exc_weight_scale=2.0, inh_weight_scale=2.5, depression_d=0.2, tau_d=500.0,
        noise_rate=5.0, noise_weight=0.007, adapt=True,
        gbar_kA_exc=0.006, gbar_kA_inh=0.004, tau_k=200.0,
        sahp_ainc_fast=0.005, sahp_tau_fast=300.0,
        sahp_ainc_slow=0.01, sahp_tau_slow=6500.0, delay_per_distance=2.0,
    ),

    # dt          integration step (ms)
    # discard_transient_ms  dropped from the front of every recording
    # target_freq           resampling rate (Hz) for the saved raster
    # participation_threshold  fraction of neurons that must fire in an event
    #                          window for it to count as a network burst
    'sim': dict(dt=0.05, discard_transient_ms=1000.0,
                target_freq=10, participation_threshold=0.35),

    # ---- how much to generate ----------------------------------------------
    'n_recordings': 50,           # per state; resumable, existing files skipped
    'n_workers': 5,               # concurrent NEURON processes
    'recording_ms': 60000.0,      # kept length per recording

    # Per-neuron Poisson streams are keyed Random123(seed, gid, recording_index).
    # The recording index is the third key, so ONE seed already gives every
    # recording a different noise realisation - no need to vary it per recording.
    'noise_seed_base': 1000,

    # 'all' every cell ~77 MB/rec | 'probe' a subset ~3 MB/rec | 'none'
    'voltage': 'probe', 'voltage_probe_n': 40, 'voltage_dt': 5.0,

    # ---- warm start ---------------------------------------------------------
    # Run the network once for warmup_ms so it settles into its natural ongoing
    # state, saving every cell's state at snapshot_times; recordings then start
    # from those. Space the snapshots by a few sahp_tau_slow (6500 ms here) so
    # they genuinely differ: 50 s in is ~8 time constants, 20 s apart is ~3.
    # EACH STATE GETS ITS OWN LIBRARY - their stationary states are not the same.
    'warmup_ms': 130000.0,
    'snapshot_times': [50000., 70000., 90000., 110000., 130000.],

    # Synaptic conductances are not restored, so each recording rebuilds them
    # from zero over ~1 s and can fire one settling burst just inside the kept
    # window. This drops that opening span. Check 1 in validation catches it if
    # the value is too small.
    'discard_extra_ms': 3000.0,
}

print("session '%s' | %s | %d rec x %.0f s per state | %d workers | voltage=%s"
      % (CONFIG['session'], CONFIG['states_to_run'], CONFIG['n_recordings'],
         CONFIG['recording_ms'] / 1000, CONFIG['n_workers'], CONFIG['voltage']))
for _s in CONFIG['states_to_run']:
    print('  %-9s sahp_ainc_slow = %.4f uS' % (_s, CONFIG['states'][_s]))


## 1. Build the network

Builds the graph from `CONFIG['topology']` and writes `_session_config.pkl` into
the session folder. Everything downstream reads that file, so the warm-start
libraries and the recordings provably used identical settings.

Cached — rerunning is free. If `CONFIG` changes after recordings exist, the next
cell tells you rather than silently mixing settings.

In [ ]:
nb.build_topology(CONFIG)


## 2. Inspect the network

The 4-panel topology view — spatial layout, hub fan-out, cluster-sorted
connection matrix, out-degree distribution — plus printed connectivity stats.
Free to look at; nothing is simulated.

In [ ]:
stats = nb.topology_report(CONFIG)


## 3. Preview each state

A short run per state with the raster and mean [K⁺]ₒ underneath, cluster-sorted
and row-randomized. Same build parameters and same total discard as the real
recordings, so what you see is representative of the kept window.

**Do this before committing hours.** Check the firing rate is sane, the bursting
looks how you expect, and [K⁺]ₒ moves. Costs a couple of minutes.

In [ ]:
nb.preview(CONFIG, duration_ms=20000.0)


## 4. Knob sweep (optional)

Burst rate against `sahp_ainc_slow` — the graded transition between your states.
Runs one short simulation per value, so pass a short `duration_ms` or few values.

In [ ]:
# nb.knob_sweep(CONFIG, values=[0.002, 0.004, 0.006, 0.008, 0.01], duration_ms=20000.0)


## 5. Preflight

Checks the interpreter, the compiled mechanisms, config drift, which libraries
exist and how much is left to do. Nothing is generated until this says OK.

In [ ]:
pre = nb.preflight(CONFIG)


## 6. Build warm-start libraries

One warm-up run per state. Skips any that already exist. States run concurrently,
so two cost about the same wall clock as one.

In [ ]:
nb.build_libraries(CONFIG, pre)
pre = nb.preflight(CONFIG)   # refresh


## 7. Generate

The long one. Resumable — rerun and it skips what already exists. Worker logs go
to `analysis/_ds_*.log`.

In [ ]:
nb.generate(CONFIG, pre)


## 8. Look at the generated rasters

The generator already writes `recordingNNN_raster.png` and
`recordingNNN_raster_shuffled.png` next to every recording; this displays a few
inline.

In [ ]:
nb.show_rasters(CONFIG, n_each=1)


## 9. Validate

Four checks, each measured against the dataset's own later behaviour rather than
any external reference:

1. **no startup burst** — zero bursts in the opening window
2. **rate is stationary** — early vs late firing rate
3. **V_rest flat from t=0** — opening Vm vs the recording's own late mean
4. **recordings differ** — distinct spike counts, confirming noise reseeding

In [ ]:
nb.validate(CONFIG)
